# Scientific ToT retrieval with PyTerrier

This notebook uses the canonical adapter, Terrier's BM25, and `pyterrier.Experiment` with `ir_measures` metrics. It compares full extracted text with available original title/Abstract sections on the same deterministic corpus and queries.

Install from the repository root:
```sh
uv sync --locked --extra pyterrier --extra notebook
uv run python -m ipykernel install --sys-prefix --name scientific-tot-pyterrier
```
Select that kernel. Set `TOT_DATA_ROOT` to restored real data (the current Hub release has queries only). A JDK must be available through `JAVA_HOME`; the documented dgx03 environment uses an isolated jdk4py runtime. Output must be a fresh directory. See `docs/pyterrier.md` for full-corpus instructions and measured results.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from scientific_tot import Dataset, DatasetAdapter
from scientific_tot.bibliography import title_abstract_from_markdown
from scientific_tot.integrations.pyterrier import (
    bm25_experiment,
    select_demo_subset,
    to_pyterrier,
)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA_ROOT = Path(
    os.environ.get(
        "TOT_DATA_ROOT",
        str(PROJECT_ROOT.parent / "multimodal-tip-of-tongue-for-scientific-papers" / "data"),
    )
)
OUTPUT = Path(os.environ.get("TOT_OUTPUT_DIR", str(PROJECT_ROOT / "results/notebook-acl-small")))
DOMAIN = "ACL"
DOCUMENTS = 250  # Set None to index the full selected corpus.
QUERIES = 10  # Set None to evaluate all selected test queries.
CORPUS_SPLIT = "train+test"  # Use "all" to include available distractors.
print(DATA_ROOT)

In [ ]:
source = Dataset(local_dir=DATA_ROOT)
adapter = DatasetAdapter(source)
queries = adapter.load_queries(domain=DOMAIN, split="test", mode="visual-and-text")
fulltext = adapter.load_corpus(
    domain=DOMAIN,
    split=CORPUS_SPLIT,
    representation="fulltext",
    missing="skip",
    include_figures=False,
    include_metadata=False,
)
print("Coverage:", len(fulltext), "emitted;", fulltext.report.skipped, "skipped")
fulltext, queries = select_demo_subset(fulltext, queries, DOCUMENTS, QUERIES)
inputs = to_pyterrier(fulltext, queries)
print(len(inputs.documents), "documents;", len(inputs.topics), "queries")
inputs.topics.head()

The real packaged metadata inspected on dgx03 has null titles and no abstracts. This **explicit** projection selects literal Markdown title and delimited Abstract text; it preserves canonical document IDs and source spans. Missing sections stay null. Nothing is summarized or generated. When authoritative bibliography fields are available, use the canonical adapter's `representation="title_abstract"` instead. A fair comparison requires identical corpus IDs; qrels must resolve in both representations.

In [ ]:
title_abstract = title_abstract_from_markdown(fulltext, missing="error")
assert {r["doc_id"] for r in title_abstract} == {r["doc_id"] for r in fulltext}
to_pyterrier(title_abstract, queries)  # Validate relevant IDs before indexing.
print(title_abstract.report.to_dict())
print(title_abstract[0]["title"])

In [ ]:
fulltext_metrics = bm25_experiment(fulltext, queries, OUTPUT / "fulltext")
bibliography_metrics = bm25_experiment(title_abstract, queries, OUTPUT / "title_abstract")
pd.concat(
    [
        fulltext_metrics.assign(representation="fulltext"),
        bibliography_metrics.assign(representation="title_abstract"),
    ],
    ignore_index=True,
)

Each representation directory contains `run.trec`, `metrics.csv`, `per-query.csv`, `topics.jsonl`, `qrels.tsv`, a reversible `query-ids.jsonl` map, and `manifest.json`. `RR@10` is MRR@10, `R@10` is Recall@10, and `R@100` is Recall@100. All metrics come from standard library implementations.

The subset includes every selected query's positive targets, then fills remaining places using sorted canonical IDs. It is a demonstration, not a full-corpus benchmark; use the documented full-split/full-corpus commands for larger experiments.

In [ ]:
import pyterrier as pt

run = pt.io.read_results(str(OUTPUT / "fulltext" / "run.trec"))
assert set(run.docno).issubset({r["doc_id"] for r in fulltext})
run[["qid", "docno", "rank", "score"]].head(10)